# Loose coupling of FreeGSNKE with TORAX: an ITER case with a physics-based transport model

Examples 12a and 12b couple TORAX to a MAST-U-like equilibrium with constant transport coefficients and fixed TORAX time steps. This notebook runs a 2 s flat-top of an ITER-like plasma (10.5 MA, 5.3 T) with

- **TORAX**: the QLKNN turbulent transport model (with constant transport inside $\rho_N = 0.3$ and in the pedestal region), a prescribed pedestal, evolving temperatures, density and current, the Newton-Raphson solver and **adaptive time stepping** (the `chi` time step calculator). Within each coupling interval TORAX takes its own time steps on the geometry interpolated in time between the equilibria at the interval ends; every TORAX step is kept in the output.
- **FreeGSNKE**: static free-boundary equilibria with an ideal **shape controller** (`torax_coupling.LinearShapeController`): the coil currents are adjusted at each solve so that the inboard/outboard boundary radii, the axis height and the X-point position keep their initial values while the profiles evolve. Without it, the large change of $\beta_p$ (from about 0.5 to 0.15 as the initial parabolic profiles relax to what QLKNN sustains) and of the internal inductance shifts the plasma inwards onto the limiter within a second, since the coil currents were optimised for the initial profiles.
- **Anderson acceleration** of the exchanged profiles (`anderson_memory=4`), which handles both the oscillatory response of the MAST-U case and the slowly converging (positive-slope) response found here.

The run takes about 30-40 minutes on a laptop (the ITER equilibrium is solved on a 129x129 grid, and each coupling interval needs a few equilibrium/transport iterations).

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torax
from freegsnke import build_machine, equilibrium_update, GSstaticsolver, torax_coupling
from freegsnke.inverse import Inverse_optimizer
from freegsnke.jtor_update import Fiesta_Topeol

## Initial ITER equilibrium

The initial equilibrium is built with the inverse solver from a set of isoflux points and null points describing an ITER-like diverted boundary (as in the ITER examples), for a Fiesta-Topeol profile; the coil currents so obtained are the starting point of the shape controller.

In [ ]:
tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/ITER/ITER_active_coils.pickle",
    passive_coils_path="../machine_configs/ITER/ITER_passive_coils.pickle",
    limiter_path="../machine_configs/ITER/ITER_limiter.pickle",
    wall_path="../machine_configs/ITER/ITER_wall.pickle",
)
eq = equilibrium_update.Equilibrium(tokamak=tokamak, Rmin=3.2, Rmax=8.8, Zmin=-5, Zmax=5, nx=129, ny=129)
fvac = 6.2 * 5.3   # R0 * B0 [T m]
profiles = Fiesta_Topeol(eq=eq, Beta0=0.5978, Ip=10.5e6, fvac=fvac, Raxis=6.2, alpha_m=2.0, alpha_n=1.395)
solver = GSstaticsolver.NKGSsolver(eq)

Rx, Zx, Ro, Zo = 5.02, -3.23, 6.34, 0.66
isoflux_set = np.array([[
    [4.25455147, 4.1881875, 4.2625, 4.45683769, 4.78746942, 5.31835938, 5.91875, 6.44275166, 6.92285156, 7.35441013, 7.73027344, 8.03046875,
     8.19517497, 8.05673414, 7.75308013, 7.37358451, 6.94355469, 6.47773438, 5.99121094, 5.49433594, Rx, 4.79042969, 4.56269531, 4.36038615],
    [0.0, 1.13554688, 2.2565134, 3.16757813, 3.80507812, 4.0499021, 3.93089003, 3.665625, 3.31076604, 2.86875, 2.3199601, 1.62832118,
     0.657421875, -0.35859375, -1.05585937, -1.59375, -2.03492727, -2.41356403, -2.75622016, -3.08699278, Zx, -2.40548044, -1.55982142, -0.67734375],
]])
constrain = Inverse_optimizer(null_points=[[Rx, Ro], [Zx, Zo]], isoflux_set=isoflux_set)
solver.inverse_solve(eq=eq, profiles=profiles, constrain=constrain, target_relative_tolerance=1e-4, target_relative_psit_update=1e-3,
                     verbose=False, l2_reg=1e-14, Picard_handover=1e-4, full_jacobian_handover=[1e-3, 1e-2])
solver.solve(eq=eq, profiles=profiles, constrain=None, target_relative_tolerance=1e-8)
print(f"R_axis = {eq.Rmagnetic():.3f} m, Z_axis = {eq.Zmagnetic():.3f} m, beta_p = {eq.poloidalBeta():.3f}")
print("shape targets (R_in, R_out, Z_axis, R_x, Z_x):", torax_coupling.boundary_targets(eq))

## TORAX configuration

The geometry section only provides the radial mesh (`n_rho`); the geometry itself comes from FreeGSNKE. The initial poloidal flux is taken from the geometry (`initial_psi_mode: 'geometry'`). The heating is 20 MW of electron heating plus the fusion power; the pedestal model prescribes 1 keV and 0.7 of the Greenwald density at $\rho_N = 0.9$.

In [ ]:
def parabolic(axis_value, edge_value, n_points=21):
    rho = np.linspace(0.0, 1.0, n_points)
    return {float(r): float(edge_value + (axis_value - edge_value) * (1 - r**2)) for r in rho}

CONFIG = {
    "plasma_composition": {"main_ion": {"D": 0.5, "T": 0.5}, "impurity": "Ne", "Z_eff": 1.6},
    "profile_conditions": {
        "Ip": 10.5e6,
        "T_i": {0.0: parabolic(8.0, 0.1)}, "T_i_right_bc": 0.1,
        "T_e": {0.0: parabolic(8.0, 0.1)}, "T_e_right_bc": 0.1,
        "n_e_right_bc_is_fGW": True, "n_e_right_bc": 0.3,
        "n_e_nbar_is_fGW": True, "nbar": 0.85, "n_e": {0: parabolic(1.4, 1.0)},
        "initial_psi_mode": "geometry",
    },
    "numerics": {
        "t_final": 2.0, "evolve_ion_heat": True, "evolve_electron_heat": True, "evolve_current": True, "evolve_density": True,
        "adaptive_dt": True, "max_dt": 0.05, "min_dt": 1e-5, "chi_timestep_prefactor": 30, "dt_reduction_factor": 3,
    },
    "geometry": {"geometry_type": "circular", "n_rho": 25, "R_major": 6.2, "a_minor": 2.0, "B_0": 5.3, "elongation_LCFS": 1.7},
    "neoclassical": {"bootstrap_current": {"bootstrap_multiplier": 1.0}},
    "sources": {
        "generic_current": {"fraction_of_total_current": 0.15, "gaussian_width": 0.075, "gaussian_location": 0.36},
        "generic_particle": {"S_total": 0.0, "deposition_location": 0.3, "particle_width": 0.25},
        "gas_puff": {"puff_decay_length": 0.3, "S_total": 0.0}, "ohmic": {},
        "pellet": {"S_total": 0.0, "pellet_width": 0.1, "pellet_deposition_location": 0.85},
        "generic_heat": {"gaussian_location": 0.127, "gaussian_width": 0.073, "P_total": 20.0e6, "electron_heat_fraction": 1.0},
        "fusion": {}, "ei_exchange": {"Qei_multiplier": 1.0},
    },
    "pedestal": {"model_name": "set_T_ped_n_ped", "set_pedestal": True, "T_i_ped": 1.0, "T_e_ped": 1.0,
                 "n_e_ped_is_fGW": True, "n_e_ped": 0.7, "rho_norm_ped_top": 0.9},
    "transport": {
        "model_name": "combined",
        "transport_models": [
            {"model_name": "constant", "rho_max": 0.3, "chi_i": 1.5, "chi_e": 1.5, "D_e": 0.25, "V_e": 0.0},
            {"model_name": "qlknn", "rho_min": 0.3, "rho_max": 0.9, "DV_effective": True, "include_ITG": True, "include_TEM": True,
             "include_ETG": True, "avoid_big_negative_s": True, "An_min": 0.05, "ITG_flux_ratio_correction": 1},
            {"model_name": "constant", "rho_min": 0.9, "chi_i": 2.0, "chi_e": 2.0, "D_e": 0.1, "V_e": 0.0},
        ],
        "chi_min": 0.05, "chi_max": 100, "D_e_min": 0.05, "D_e_max": 50, "V_e_min": -10, "V_e_max": 10,
        "smoothing_zones": [{"rho_min": 0.3, "rho_max": 0.9, "smoothing_width": 0.1}],
    },
    "solver": {"solver_type": "newton_raphson", "use_predictor_corrector": True, "n_corrector_steps": 10,
               "chi_pereverzev": 30, "D_pereverzev": 15, "use_pereverzev": True},
    "time_step_calculator": {"calculator_type": "chi"},
}
torax_config = torax.ToraxConfig.from_dict(CONFIG)

## Equilibrium provider with shape control

`LinearShapeController` builds the response matrix of the shape targets to the coil currents by finite differences (one forward solve per coil, repeated after every accepted coupling interval) and, at each equilibrium solve, applies Gauss-Newton current updates until the targets are met to 2 mm.

In [ ]:
coils = list(tokamak.coils_list[: tokamak.n_active_coils])
shape_controller = torax_coupling.LinearShapeController(coils, tolerance=2e-3, relinearise_every=1)
equilibrium_solver = torax_coupling.StaticEquilibriumSolver(
    eq, profiles, solver=solver, target_relative_tolerance=1e-7, shape_controller=shape_controller
)

## Coupled run

The residual is the relative change, between successive iterations, of the toroidal current density $J_\phi = R p' + FF'/(\mu_0 R)$ evaluated at the inboard and outboard radii of each flux surface; the iteration stops below `tolerance`.

In [ ]:
t0 = time.time()
result = torax_coupling.run_loose_coupling(
    torax_config, equilibrium_solver, coupling_dt=0.1, max_iterations=10, tolerance=1e-3,
    relaxation=0.5, anderson_memory=4, initial_iterations=1, verbose=True,
)
print(f"wall time {time.time() - t0:.0f} s, TORAX error state: {result.sim_error}")
print("iterations per coupling interval:", result.iterations)
print("TORAX steps per coupling interval:", result.torax_substeps)

## Results

In [ ]:
profiles_out = result.torax_output["profiles"]
scalars = result.torax_output["scalars"]
t_torax = profiles_out["time"].values
rho, rho_face = profiles_out["rho_norm"].values, profiles_out["rho_face_norm"].values

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes[0, 0].plot(t_torax, profiles_out["T_e"].values[:, 0], label="$T_e(0)$")
axes[0, 0].plot(t_torax, profiles_out["T_i"].values[:, 0], label="$T_i(0)$")
axes[0, 0].set_ylabel("[keV]"); axes[0, 0].legend()
for t in result.times[::4]:
    i = int(np.argmin(np.abs(t_torax - t)))
    axes[0, 1].plot(rho, profiles_out["T_e"].values[i], label=f"t = {t:.1f} s")
    axes[1, 0].plot(rho_face, profiles_out["q"].values[i])
axes[0, 1].set_ylabel("$T_e$ [keV]"); axes[0, 1].legend(fontsize=8); axes[1, 0].set_ylabel("q")
for ax in (axes[0, 1], axes[1, 0]):
    ax.set_xlabel(r"$\rho_N$")
for k, r in enumerate(result.residuals[1:]):
    axes[1, 1].semilogy(np.arange(1, len(r) + 1), r, "-o", ms=3, color=plt.cm.viridis(k / (len(result.residuals) - 2)))
axes[1, 1].axhline(1e-3, ls="--", color="k"); axes[1, 1].set_xlabel("coupling iteration"); axes[1, 1].set_ylabel("residual (colour: time)")
axes[0, 0].set_xlabel("t [s]")
plt.tight_layout()

In [ ]:
# equilibrium at the end of the run: the shape is held by the controller
final = result.equilibrium_ids[-1].time_slice[0]
print("final shape targets:", torax_coupling.boundary_targets(eq))
print(f"beta_p = {eq.poloidalBeta():.3f}, R_axis = {eq.Rmagnetic():.3f} m, largest shape error = {max(shape_controller.history):.1e} m")
fig, ax = plt.subplots(figsize=(5, 8))
eq.plot(axis=ax, show=False)
plt.show()